# keyboard_config

> Alignment-specific keyboard building blocks for assembly into a shared ZoneManager

In [ ]:
#| default_exp components.keyboard_config

In [ ]:
#| export
from typing import Tuple

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction

# Card stack library
from cjm_fasthtml_card_stack.core.config import CardStackConfig
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds
from cjm_fasthtml_card_stack.core.button_ids import CardStackButtonIds
from cjm_fasthtml_card_stack.keyboard.actions import (
    create_card_stack_focus_zone, create_card_stack_nav_actions,
)

## Keyboard Parts Builder

Returns `(zone, actions, modes)` tuple for assembly into a shared `ZoneManager`
by the combined-level keyboard config. No sub-modes needed (unlike decomp's split mode).

In [ ]:
#| export
def create_align_kb_parts(
    ids:CardStackHtmlIds,  # Card stack HTML IDs
    button_ids:CardStackButtonIds,  # Card stack button IDs for navigation
    config:CardStackConfig,  # Card stack configuration
) -> Tuple[FocusZone, tuple, tuple]:  # (zone, actions, modes)
    """Create alignment-specific keyboard building blocks."""
    # Card stack zone from library.
    # Note: No data_attributes needed — onAlignFocusChange callback manually updates the
    # card stack hidden input and reads start/end times directly from card DOM elements.
    # The `label` drives per-zone section headers in the keyboard-hints modal
    # (cjm-fasthtml-keyboard-navigation 0.0.22+) — appears as
    # "VAD Alignment — Navigation" / "VAD Alignment — Audio" / etc. when this manager
    # is merged with segmentation's into segment-align's combined ZoneManager.
    card_zone = create_card_stack_focus_zone(
        ids=ids,
        on_focus_change="onAlignFocusChange",
        label="VAD Alignment",
    )

    # Card stack navigation actions from library (arrows, page jump, first/last, width)
    nav_actions = create_card_stack_nav_actions(
        zone_id=card_zone.id,
        button_ids=button_ids,
        config=config,
    )

    # Replay action — Space key to replay current chunk's audio
    replay_action = KeyAction(
        key=" ",  # Space
        js_callback="replayAlignSegment",
        zone_ids=(card_zone.id,),
        description="Replay audio",
        hint_group="Audio",
    )

    # Auto-play toggle action — A key toggles auto-navigate, syncs checkbox + colors
    auto_play_action = KeyAction(
        key="a",
        js_callback="toggleAlignAutoPlay",
        zone_ids=(card_zone.id,),
        description="Toggle auto-play",
        hint_group="Audio",
    )

    # Playback speed cycle — Shift+ArrowUp/Down steps through PLAYBACK_SPEEDS.
    # Delegates to cjm-fasthtml-web-audio's cycle{Ns}Speed{Up,Down} helpers, which
    # update the shared <select>, JS playbackSpeed, and fire HTMX persistence in one go.
    speed_up_action = KeyAction(
        key="ArrowUp",
        modifiers=frozenset({"shift"}),
        js_callback="cycleAlignSpeedUp",
        zone_ids=(card_zone.id,),
        description="Speed up",
        hint_group="Audio",
    )
    speed_down_action = KeyAction(
        key="ArrowDown",
        modifiers=frozenset({"shift"}),
        js_callback="cycleAlignSpeedDown",
        zone_ids=(card_zone.id,),
        description="Slow down",
        hint_group="Audio",
    )

    # Combine navigation + audio actions
    actions = nav_actions + (replay_action, auto_play_action, speed_up_action, speed_down_action)
    modes = ()  # No sub-modes — alignment uses a single navigation mode

    return card_zone, actions, modes

In [ ]:
# Speed cycle KeyActions wire correctly into the alignment zone
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds as _IDs
from cjm_fasthtml_card_stack.core.button_ids import CardStackButtonIds as _BtnIds
from cjm_fasthtml_card_stack.core.config import CardStackConfig as _Cfg

_zone, _actions, _modes = create_align_kb_parts(
    ids=_IDs(prefix="align-cs"),
    button_ids=_BtnIds(prefix="align-cs"),
    config=_Cfg(),
)

_by_key = {(a.key, frozenset(a.modifiers)): a for a in _actions}

_up = _by_key.get(("ArrowUp", frozenset({"shift"})))
_dn = _by_key.get(("ArrowDown", frozenset({"shift"})))
assert _up is not None, "Shift+ArrowUp binding missing"
assert _dn is not None, "Shift+ArrowDown binding missing"

# Scoped to the alignment card-stack zone, wired to web-audio's cycle helpers,
# grouped with the other audio bindings in the hint panel
assert _up.js_callback == "cycleAlignSpeedUp"
assert _dn.js_callback == "cycleAlignSpeedDown"
assert _up.zone_ids == (_zone.id,)
assert _dn.zone_ids == (_zone.id,)
assert _up.hint_group == "Audio"
assert _dn.hint_group == "Audio"

# Plain ArrowUp/ArrowDown (card-stack nav) must NOT collide — distinct modifiers frozenset
_nav_up = _by_key.get(("ArrowUp", frozenset()))
_nav_dn = _by_key.get(("ArrowDown", frozenset()))
assert _nav_up is not None and _nav_up.htmx_trigger, "Plain ArrowUp should still be the card-stack nav binding"
assert _nav_dn is not None and _nav_dn.htmx_trigger, "Plain ArrowDown should still be the card-stack nav binding"

print("Shift+Arrow speed cycle bindings test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()